🔹 Step 1: Environment Setup

In [190]:
# Install required libraries (run once)
# pip install pandas numpy matplotlib scikit-learn tensorflow

In [191]:
!python --version

Python 3.12.12


🔹 Step 2: Import Libraries

In [192]:
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout, Input, Bidirectional
from sklearn.utils.class_weight import compute_class_weight
from tensorflow.keras.callbacks import EarlyStopping

🔹 Step 3: Load & Inspect Dataset

In [193]:
df = pd.read_csv("Job_3_Resource_sentiment.csv")
print(df.columns)

Index(['2401', 'Borderlands', 'Positive',
       'im getting on borderlands and i will murder you all ,'],
      dtype='object')


In [194]:
df.head()

,2401,Borderlands,Positive,"im getting on borderlands and i will murder you all ,"
0,2401,Borderlands,Positive,I am coming to the borders and I will kill you...
1,2401,Borderlands,Positive,im getting on borderlands and i will kill you ...
2,2401,Borderlands,Positive,im coming on borderlands and i will murder you...
3,2401,Borderlands,Positive,im getting on borderlands 2 and i will murder ...
4,2401,Borderlands,Positive,im getting into borderlands and i can murder y...


In [195]:
df.columns

Index(['2401', 'Borderlands', 'Positive',
       'im getting on borderlands and i will murder you all ,'],
      dtype='object')

In [196]:
df.rename(columns={'Positive': 'sentiment'}, inplace=True)
df.rename(columns={'im getting on borderlands and i will murder you all ,': 'text'}, inplace=True)

In [197]:
df.columns

Index(['2401', 'Borderlands', 'sentiment', 'text'], dtype='object')

In [198]:
df.sample(5)

,2401,Borderlands,sentiment,text
34381,6705,Fortnite,Irrelevant,Jennifer Walters Is SECRETLY Reactive! (How is...
40367,1328,Battlefield,Positive,almost done YES YES tomorrow we can finish ou...
19120,12475,WorldOfCraft,Positive,"Well, extra leech and water sure was fun while..."
55659,2355,CallOfDuty,Neutral,And So I'i m happily going on about the game w...
69087,3834,Cyberpunk2077,Positive,Congrats to the complete team! look forward fu...


In [199]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 74681 entries, 0 to 74680
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   2401         74681 non-null  int64 
 1   Borderlands  74681 non-null  object
 2   sentiment    74681 non-null  object
 3   text         73995 non-null  object
dtypes: int64(1), object(3)
memory usage: 2.3+ MB


In [200]:
df = df[['text', 'sentiment']]

In [201]:
df.sample(5)

,text,sentiment
69503,"I love all signs, neons in",Positive
63908,@EAMaddenNFL please add the Buffalo Bills 90's...,Negative
14202,Feeling good as a good man - Witch doctor,Positive
20369,"""...the movie will wipe the slate clean and st...",Positive
72688,A company that has the ability to develop and ...,Negative


In [202]:
df

,text,sentiment
0,I am coming to the borders and I will kill you...,Positive
1,im getting on borderlands and i will kill you ...,Positive
2,im coming on borderlands and i will murder you...,Positive
3,im getting on borderlands 2 and i will murder ...,Positive
4,im getting into borderlands and i can murder y...,Positive
...,...,...
74676,Just realized that the Windows partition of my...,Positive
74677,Just realized that my Mac window partition is ...,Positive
74678,Just realized the windows partition of my Mac ...,Positive
74679,Just realized between the windows partition of...,Positive


In [203]:
df.shape

(74681, 2)

In [204]:
df.isnull().sum()

,0
text,686
sentiment,0


In [205]:
print(df['sentiment'].value_counts())

sentiment
Negative      22542
Positive      20831
Neutral       18318
Irrelevant    12990
Name: count, dtype: int64


In [206]:
df.duplicated().sum()

np.int64(4909)

🔹 Step 4: Data Cleaning

In [207]:
df.dropna(inplace=True)

In [208]:
df.isnull().sum()

,0
text,0
sentiment,0


In [209]:
df['text'] = df['text'].astype(str)

In [210]:
df['text']

,text
0,I am coming to the borders and I will kill you...
1,im getting on borderlands and i will kill you ...
2,im coming on borderlands and i will murder you...
3,im getting on borderlands 2 and i will murder ...
4,im getting into borderlands and i can murder y...
...,...
74676,Just realized that the Windows partition of my...
74677,Just realized that my Mac window partition is ...
74678,Just realized the windows partition of my Mac ...
74679,Just realized between the windows partition of...


In [211]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r"http\S+", "", text)
    text = re.sub(r"[^a-z\s]", "", text)
    return text.strip()

df['text'] = df['text'].apply(clean_text)

In [212]:
df['text']

,text
0,i am coming to the borders and i will kill you...
1,im getting on borderlands and i will kill you all
2,im coming on borderlands and i will murder you...
3,im getting on borderlands and i will murder y...
4,im getting into borderlands and i can murder y...
...,...
74676,just realized that the windows partition of my...
74677,just realized that my mac window partition is ...
74678,just realized the windows partition of my mac ...
74679,just realized between the windows partition of...


🔹 Step 5: Encode Target Labels

In [213]:
encoder = LabelEncoder()
df['sentiment_encoded'] = encoder.fit_transform(df['sentiment'])

print("Label mapping:")
for i, c in enumerate(encoder.classes_):
    print(i, "->", c)

Label mapping:
0 -> Irrelevant
1 -> Negative
2 -> Neutral
3 -> Positive


In [214]:
encoder.classes_

array(['Irrelevant', 'Negative', 'Neutral', 'Positive'], dtype=object)

In [215]:
df.sample(10)

,text,sentiment,sentiment_encoded
31028,i dont know but hylissang looks kind of unfort...,Neutral,2
55329,it imagine thinking like this so fucking stupi...,Irrelevant,0
27901,minutes to download apex legends wtf,Negative,1
21569,best csgo trading websites list csbetsorgcsg...,Neutral,2
9291,global game gotta show love to we all the,Positive,3
30635,permabanned on my team or not hes getting bann...,Irrelevant,0
59653,i dont care what your political beliefs are it...,Irrelevant,0
37001,i need microsoft to hurry up and release those...,Negative,1
39256,out some reason wild roots became epic in its ...,Neutral,2
56476,rainbowgame first of all i love this game but ...,Neutral,2


🔹 Step 6: Text Tokenization & Padding

In [216]:
max_words = 20000
max_length = 100

tokenizer = Tokenizer(num_words=max_words, oov_token="<OOV>")
tokenizer.fit_on_texts(df['text'])

X = pad_sequences(
    tokenizer.texts_to_sequences(df['text']),
    maxlen=max_length,
    padding='post'
)

y = df['sentiment_encoded'].values

In [217]:
tokenizer

In [218]:
df

,text,sentiment,sentiment_encoded
0,i am coming to the borders and i will kill you...,Positive,3
1,im getting on borderlands and i will kill you all,Positive,3
2,im coming on borderlands and i will murder you...,Positive,3
3,im getting on borderlands and i will murder y...,Positive,3
4,im getting into borderlands and i can murder y...,Positive,3
...,...,...,...
74676,just realized that the windows partition of my...,Positive,3
74677,just realized that my mac window partition is ...,Positive,3
74678,just realized the windows partition of my mac ...,Positive,3
74679,just realized between the windows partition of...,Positive,3


In [219]:
X.shape

(73995, 100)

In [220]:
X.dtype

dtype('int32')

In [221]:
y.shape

(73995,)

In [222]:
y.dtype

dtype('int64')

In [223]:
X.ndim

2

In [224]:
y.ndim

1

In [225]:
X.nbytes

29598000

In [226]:
y.nbytes

591960

In [227]:
X

array([[   3,  101,  377, ...,    0,    0,    0],
       [  31,  158,   14, ...,    0,    0,    0],
       [  31,  377,   14, ...,    0,    0,    0],
       ...,
       [  22, 1837,    2, ...,    0,    0,    0],
       [  22, 1837,  693, ...,    0,    0,    0],
       [  22,   32,    2, ...,    0,    0,    0]], dtype=int32)

In [228]:
y

array([3, 3, 3, ..., 3, 3, 3])